## Behaviour if grouping and calculating the mean if NaN-values are in the group

In [1]:
import pandas as pd
import numpy as np
import holidays
holidays_de= holidays.Germany()

Extra code run to make holidays run:
import sys
!{sys.executable} -m pip install --force-reinstall holidays
!{sys.executable} -m pip show holidays


In [2]:
# Load merged hourly dataset
df = pd.read_csv("../../data_cleaned/merged/02_2_Data_imputed_2019_to_2025.csv")
# Convert timestamp columns to datetime
df["period_start_utc"] = pd.to_datetime(df["period_start_utc"], utc=True, errors="coerce")
df["period_end_utc"] = pd.to_datetime(df["period_end_utc"], utc=True, errors="coerce")
df["date"] = pd.to_datetime(df["date"], errors="coerce")

df = df.sort_values("period_start_utc")

df.head()

,date,year,month,day,dayofyear,hour,week,dayofweek,price,period_start_utc,...,on_wind_da,on_wind_act,solar_da,solar_act,gen_forecast_da,gen_actual,res_sum_da,res_sum_act,imputed,interpolated
0,2019-01-01,2019,1,1,1,0,1,1,10.07,2019-01-01 00:00:00+00:00,...,20626.5625,22315.6300,0.0,0.1350,51084.26,53141.07,25669.0050,25183.9800,0,0
1,2019-01-01,2019,1,1,1,1,1,1,-4.08,2019-01-01 01:00:00+00:00,...,22355.6850,23193.3725,0.0,0.1225,51512.67,52534.02,27384.1025,25653.8950,0,0
2,2019-01-01,2019,1,1,1,2,1,1,-9.91,2019-01-01 02:00:00+00:00,...,24032.4775,24508.7150,0.0,0.1250,52693.45,53217.19,29010.1275,27205.4650,0,0
3,2019-01-01,2019,1,1,1,3,1,1,-7.41,2019-01-01 03:00:00+00:00,...,25452.1675,26373.4375,0.0,0.1300,53666.46,54735.54,30359.5675,28951.3075,0,0
4,2019-01-01,2019,1,1,1,4,1,1,-12.55,2019-01-01 04:00:00+00:00,...,26526.3700,27931.3025,0.0,0.1275,54161.85,55846.36,31409.2100,30308.7475,0,0


In [3]:

def add_fourier(df, col, period, K=1, drop=False, offset=0):
    # offset=1 is useful for 1-based columns like month/dayofyear/week
    x = df[col].astype(float) - offset
    for k in range(1, K + 1):
        df[f"{col}_sin{k}"] = np.sin(2 * np.pi * k * x / period)
        df[f"{col}_cos{k}"] = np.cos(2 * np.pi * k * x / period)
    if drop:
        df.drop(columns=[col], inplace=True)
    return df

# Example periods
# hour: 0-23 -> P=24, offset=0
# dayofweek: 0-6 or 1-7 -> P=7 (set offset accordingly)
# month: 1-12 -> P=12, offset=1
# dayofyear: 1-365/366 -> usually P=365.25, offset=1
# week: 1-52/53 -> P=52.18 (approx), offset=1

#df = add_fourier(df, "year", 24, K=1, offset=0)
df = add_fourier(df, "dayofyear", 365.25, K=1, offset=1)
df = add_fourier(df, "hour", 24, K=1, offset=0)

df = add_fourier(df, "dayofweek", 7, K=1, offset=0)   # change offset if 1..7
#df = add_fourier(df, "month", 12, K=1, offset=1)
#df = add_fourier(df, "week", 52.18, K=1, offset=1)


In [4]:
df.columns

Index(['date', 'year', 'month', 'day', 'dayofyear', 'hour', 'week',
       'dayofweek', 'price', 'period_start_utc', 'period_end_utc', 'c_by_hour',
       'load_forecast_da', 'load_actual', 'off_wind_da', 'off_wind_act',
       'on_wind_da', 'on_wind_act', 'solar_da', 'solar_act', 'gen_forecast_da',
       'gen_actual', 'res_sum_da', 'res_sum_act', 'imputed', 'interpolated',
       'dayofyear_sin1', 'dayofyear_cos1', 'hour_sin1', 'hour_cos1',
       'dayofweek_sin1', 'dayofweek_cos1'],
      dtype='object')

In [5]:
df.head(10)

,date,year,month,day,dayofyear,hour,week,dayofweek,price,period_start_utc,...,res_sum_da,res_sum_act,imputed,interpolated,dayofyear_sin1,dayofyear_cos1,hour_sin1,hour_cos1,dayofweek_sin1,dayofweek_cos1
0,2019-01-01,2019,1,1,1,0,1,1,10.07,2019-01-01 00:00:00+00:00,...,25669.0050,25183.9800,0,0,0.0,1.0,0.000000,1.000000e+00,0.781831,0.62349
1,2019-01-01,2019,1,1,1,1,1,1,-4.08,2019-01-01 01:00:00+00:00,...,27384.1025,25653.8950,0,0,0.0,1.0,0.258819,9.659258e-01,0.781831,0.62349
2,2019-01-01,2019,1,1,1,2,1,1,-9.91,2019-01-01 02:00:00+00:00,...,29010.1275,27205.4650,0,0,0.0,1.0,0.500000,8.660254e-01,0.781831,0.62349
3,2019-01-01,2019,1,1,1,3,1,1,-7.41,2019-01-01 03:00:00+00:00,...,30359.5675,28951.3075,0,0,0.0,1.0,0.707107,7.071068e-01,0.781831,0.62349
4,2019-01-01,2019,1,1,1,4,1,1,-12.55,2019-01-01 04:00:00+00:00,...,31409.2100,30308.7475,0,0,0.0,1.0,0.866025,5.000000e-01,0.781831,0.62349
5,2019-01-01,2019,1,1,1,5,1,1,-17.25,2019-01-01 05:00:00+00:00,...,32935.1225,31651.2950,0,0,0.0,1.0,0.965926,2.588190e-01,0.781831,0.62349
6,2019-01-01,2019,1,1,1,6,1,1,-15.07,2019-01-01 06:00:00+00:00,...,33738.3125,32460.5825,0,0,0.0,1.0,1.000000,6.123234e-17,0.781831,0.62349
7,2019-01-01,2019,1,1,1,7,1,1,-4.93,2019-01-01 07:00:00+00:00,...,34253.8150,33274.8775,0,0,0.0,1.0,0.965926,-2.588190e-01,0.781831,0.62349
8,2019-01-01,2019,1,1,1,8,1,1,-6.33,2019-01-01 08:00:00+00:00,...,35431.8175,33572.5925,0,0,0.0,1.0,0.866025,-5.000000e-01,0.781831,0.62349
9,2019-01-01,2019,1,1,1,9,1,1,-4.93,2019-01-01 09:00:00+00:00,...,36845.7375,35221.0250,0,0,0.0,1.0,0.707107,-7.071068e-01,0.781831,0.62349


### Add holidays

In [6]:
df['is_holiday']= df ['date'].apply(lambda x: 1 if x in holidays_de else 0)

In [7]:
df['day_type']= 'weekday'
df.loc[df['dayofweek'].isin([5,6]), 'day_type'] = 'weekend'
df.loc[df['is_holiday'] == 1, 'day_type'] = 'holiday'
df.head()

,date,year,month,day,dayofyear,hour,week,dayofweek,price,period_start_utc,...,imputed,interpolated,dayofyear_sin1,dayofyear_cos1,hour_sin1,hour_cos1,dayofweek_sin1,dayofweek_cos1,is_holiday,day_type
0,2019-01-01,2019,1,1,1,0,1,1,10.07,2019-01-01 00:00:00+00:00,...,0,0,0.0,1.0,0.000000,1.000000,0.781831,0.62349,1,holiday
1,2019-01-01,2019,1,1,1,1,1,1,-4.08,2019-01-01 01:00:00+00:00,...,0,0,0.0,1.0,0.258819,0.965926,0.781831,0.62349,1,holiday
2,2019-01-01,2019,1,1,1,2,1,1,-9.91,2019-01-01 02:00:00+00:00,...,0,0,0.0,1.0,0.500000,0.866025,0.781831,0.62349,1,holiday
3,2019-01-01,2019,1,1,1,3,1,1,-7.41,2019-01-01 03:00:00+00:00,...,0,0,0.0,1.0,0.707107,0.707107,0.781831,0.62349,1,holiday
4,2019-01-01,2019,1,1,1,4,1,1,-12.55,2019-01-01 04:00:00+00:00,...,0,0,0.0,1.0,0.866025,0.500000,0.781831,0.62349,1,holiday


In [8]:
df['day_type'].value_counts()

day_type
weekday    42575
weekend    17280
holiday     1512
Name: count, dtype: int64

In [9]:
df.to_csv("../../data_cleaned/merged/02_3_Data_imputed_2019_to_2025_with_FH.csv", index=False)